# Deep Learning – Assignment 2
**STL-10 Dataset | 96×96 color images | 10 classes**

| Experiment | Description |
|---|---|
| A | CNN from Scratch (VGG_SmallSigmoid) |
| B | Transfer Learning — ResNet-50 & VGG-16 (frozen + full fine-tune) |
| C | Semi-Supervised — ConvAutoencoder pretraining + Classifier fine-tuning |

## Imports & Global Config

In [ ]:
import gc
import os
import sys
import time
import tarfile
from datetime import datetime, timedelta
from typing import Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

if sys.version_info >= (3, 0, 0):
    import urllib.request as urllib_req
else:
    import urllib as urllib_req

DATA_DIR         = "./data/stl10_binary"
TRAIN_X_PATH     = os.path.join(DATA_DIR, "train_X.bin")
TRAIN_Y_PATH     = os.path.join(DATA_DIR, "train_y.bin")
TEST_X_PATH      = os.path.join(DATA_DIR, "test_X.bin")
TEST_Y_PATH      = os.path.join(DATA_DIR, "test_y.bin")
UNLABELED_X_PATH = os.path.join(DATA_DIR, "unlabeled_X.bin")
DATA_URL         = "http://ai.stanford.edu/~acoates/stl10/stl10_binary.tar.gz"
RESULTS_FILE     = "model_comparison.xlsx"

NUM_CLASSES = 10
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP     = DEVICE.type == "cuda"

STL10_MEAN    = [0.4467, 0.4398, 0.4066]
STL10_STD     = [0.2603, 0.2565, 0.2712]
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

CLASS_NAMES = ["airplane", "bird", "car", "deer", "dog",
               "horse", "monkey", "ship", "truck", "frog"]

print(f"Device: {DEVICE}  |  AMP: {USE_AMP}")

## STL-10 Binary Reader

In [ ]:
def download_and_extract() -> None:
    dest = "./data"
    os.makedirs(dest, exist_ok=True)
    filename = DATA_URL.split("/")[-1]
    filepath = os.path.join(dest, filename)
    if not os.path.exists(filepath):
        def _progress(count: int, block_size: int, total_size: int) -> None:
            sys.stdout.write(
                "\rDownloading %s  %.2f%%" % (
                    filename, float(count * block_size) / float(total_size) * 100.0
                )
            )
            sys.stdout.flush()
        filepath, _ = urllib_req.urlretrieve(DATA_URL, filepath, reporthook=_progress)
        print("\nDownloaded", filename)
        tarfile.open(filepath, "r:gz").extractall(dest)


def read_all_images(path: str) -> np.ndarray:
    with open(path, "rb") as f:
        data = np.fromfile(f, dtype=np.uint8)
    images = np.reshape(data, (-1, 3, 96, 96))
    return np.transpose(images, (0, 3, 2, 1))   # (N, 96, 96, 3)


def read_labels(path: str) -> np.ndarray:
    with open(path, "rb") as f:
        return np.fromfile(f, dtype=np.uint8)


download_and_extract()
print("Dataset ready.")

## Excel Results Logger

In [ ]:
_HEADER_FILL = PatternFill("solid", fgColor="1F4E79")
_STAGE_FILLS = {
    "A": PatternFill("solid", fgColor="D9E1F2"),
    "B": PatternFill("solid", fgColor="E2EFDA"),
    "C": PatternFill("solid", fgColor="FFF2CC"),
}
_THIN_BORDER = Border(
    left=Side(style="thin"), right=Side(style="thin"),
    top=Side(style="thin"),  bottom=Side(style="thin"),
)
_HEADERS    = ["Timestamp", "Stage", "Architecture", "Mode", "Optimizer",
               "LR", "Epochs Trained", "Train Acc (%)", "Test Acc (%)", "Notes"]
_COL_WIDTHS = [20, 8, 18, 20, 12, 10, 16, 15, 15, 40]


def _apply_header_style(ws) -> None:
    for col_idx, header in enumerate(_HEADERS, start=1):
        cell = ws.cell(row=1, column=col_idx, value=header)
        cell.font      = Font(bold=True, color="FFFFFF", size=11)
        cell.fill      = _HEADER_FILL
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border    = _THIN_BORDER
        ws.column_dimensions[get_column_letter(col_idx)].width = _COL_WIDTHS[col_idx - 1]
    ws.row_dimensions[1].height = 30
    ws.freeze_panes = "A2"


def log_result(
    stage: str,
    architecture: str,
    mode: str,
    optimizer: str,
    lr: float,
    epochs_trained: int,
    train_acc: Optional[float],
    test_acc: Optional[float],
    notes: str = "",
) -> None:
    if os.path.exists(RESULTS_FILE):
        wb = load_workbook(RESULTS_FILE)
        ws = wb.active
    else:
        wb = Workbook()
        ws = wb.active
        ws.title = "Results"
        _apply_header_style(ws)

    next_row      = ws.max_row + 1
    train_acc_pct = round(train_acc * 100, 2) if train_acc is not None else "N/A"
    test_acc_pct  = round(test_acc  * 100, 2) if test_acc  is not None else "N/A"

    row_values = [
        datetime.now().strftime("%Y-%m-%d %H:%M"),
        stage, architecture, mode, optimizer, lr,
        epochs_trained, train_acc_pct, test_acc_pct, notes,
    ]
    fill = _STAGE_FILLS.get(stage, PatternFill())
    for col_idx, value in enumerate(row_values, start=1):
        cell           = ws.cell(row=next_row, column=col_idx, value=value)
        cell.fill      = fill
        cell.border    = _THIN_BORDER
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

    wb.save(RESULTS_FILE)
    print(f"  [Excel] Row logged → {RESULTS_FILE}  (row {next_row})")


print("Logger ready.")

## Shared Utilities

In [ ]:
class EarlyStopping:
    def __init__(self, patience: int = 7, min_delta: float = 1e-4) -> None:
        self.patience  = patience
        self.min_delta = min_delta
        self.counter   = 0
        self.best_loss = float("inf")

    def __call__(self, val_loss: float) -> bool:
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter   = 0
        else:
            self.counter += 1
        return self.counter >= self.patience


def _separator(title: str) -> None:
    line = "=" * 70
    print(f"\n{line}\n  {title}\n{line}\n")


def _clear_gpu() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        allocated = torch.cuda.memory_allocated() / 1024 ** 2
        print(f"  GPU memory after cleanup: {allocated:.1f} MB allocated")


print("Utilities ready.")

---
## Experiment A — CNN from Scratch
**Architecture:** VGG_SmallSigmoid — 4-block VGG backbone + Sigmoid classifier head  
**Optimizer:** SGD + Nesterov momentum  
**Scheduler:** CosineAnnealingLR  
**Training data:** 5,000 labeled STL-10 images

In [ ]:
class STL10DatasetA(Dataset):
    def __init__(self, images: np.ndarray, labels: np.ndarray, transform=None) -> None:
        self.images    = images
        self.labels    = labels.astype(np.int64) - 1
        self.transform = transform

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        img = torch.from_numpy(self.images[idx]).permute(2, 0, 1).float() / 255.0
        if self.transform:
            img = self.transform(img)
        return img, int(self.labels[idx])


def _build_transforms_a(train: bool) -> transforms.Compose:
    norm = transforms.Normalize(mean=STL10_MEAN, std=STL10_STD)
    if train:
        return transforms.Compose([
            transforms.RandomCrop(96, padding=8),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
            norm,
        ])
    return transforms.Compose([norm])


def _load_labeled_a(images_path: str, labels_path: str, train: bool, batch_size: int) -> DataLoader:
    images  = read_all_images(images_path)
    labels  = read_labels(labels_path)
    dataset = STL10DatasetA(images, labels, transform=_build_transforms_a(train))
    return DataLoader(dataset, batch_size=batch_size, shuffle=train, num_workers=2, pin_memory=True)


class VGGSmallSigmoid(nn.Module):
    """
    4-block VGG backbone with AdaptiveAvgPool so it is input-size agnostic.
    Classifier head: 512 -> 64 (Sigmoid) -> 32 (ReLU+BN) -> 10.
    Best architecture from HW1, ported to STL-10 (96x96).
    """

    def __init__(self, num_classes: int = 10) -> None:
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(3,   64,  3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(),
            nn.Conv2d(64,  64,  3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64,  128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.fc1        = nn.Linear(512, 64)
        self.drop1      = nn.Dropout(0.1)
        self.fc2        = nn.Linear(64, 32)
        self.bn2        = nn.BatchNorm1d(32)
        self.drop2      = nn.Dropout(0.2)
        self.classifier = nn.Linear(32, num_classes)
        nn.init.xavier_uniform_(self.fc1.weight);  nn.init.constant_(self.fc1.bias, 0)
        nn.init.kaiming_normal_(self.fc2.weight);  nn.init.constant_(self.fc2.bias, 0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.backbone(x).view(x.size(0), -1)
        x = self.drop1(torch.sigmoid(self.fc1(x)))
        x = self.drop2(torch.relu(self.bn2(self.fc2(x))))
        return self.classifier(x)


def _train_epoch_a(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    scaler: torch.cuda.amp.GradScaler,
) -> Tuple[float, float]:
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            out  = model(images)
            loss = criterion(out, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * images.size(0)
        correct    += (out.argmax(1) == labels).sum().item()
        total      += images.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def _eval_a(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
) -> Tuple[float, float]:
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            out  = model(images)
            loss = criterion(out, labels)
        total_loss += loss.item() * images.size(0)
        correct    += (out.argmax(1) == labels).sum().item()
        total      += images.size(0)
    return total_loss / total, correct / total


print("CNN from Scratch — classes defined.")

In [ ]:
def run_cnn_from_scratch() -> None:
    BATCH_SIZE   = 64
    NUM_EPOCHS   = 50
    LR           = 1e-2
    WEIGHT_DECAY = 1e-4
    PATIENCE     = 10

    train_loader = _load_labeled_a(TRAIN_X_PATH, TRAIN_Y_PATH, train=True,  batch_size=BATCH_SIZE)
    test_loader  = _load_labeled_a(TEST_X_PATH,  TEST_Y_PATH,  train=False, batch_size=BATCH_SIZE)
    print(f"Train: {len(train_loader.dataset)} images  |  Test: {len(test_loader.dataset)} images")

    model     = VGGSmallSigmoid(NUM_CLASSES).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=LR, momentum=0.9, nesterov=True, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    scaler    = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    stopper   = EarlyStopping(patience=PATIENCE)

    best_acc, final_train_acc, epochs_trained = 0.0, 0.0, 0
    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss, train_acc = _train_epoch_a(model, train_loader, criterion, optimizer, scaler)
        test_loss,  test_acc  = _eval_a(model, test_loader, criterion)
        scheduler.step()
        epochs_trained  = epoch
        final_train_acc = train_acc
        if test_acc > best_acc:
            best_acc = test_acc
            torch.save(model.state_dict(), "cnn_scratch_best.pth")
        print(
            f"Epoch [{epoch:>2}/{NUM_EPOCHS}]  "
            f"Train loss: {train_loss:.4f}  acc: {train_acc*100:.2f}%  |  "
            f"Test  loss: {test_loss:.4f}  acc: {test_acc*100:.2f}%"
        )
        if stopper(test_loss):
            print(f"\nEarly stopping at epoch {epoch}")
            break

    print(f"\nBest test accuracy (CNN from Scratch): {best_acc*100:.2f}%")
    log_result(
        stage="A", architecture="VGG_SmallSigmoid", mode="scratch",
        optimizer="SGD", lr=LR, epochs_trained=epochs_trained,
        train_acc=final_train_acc, test_acc=best_acc,
        notes="4-block VGG backbone + Sigmoid head, CosineAnnealingLR, EarlyStopping",
    )
    del model, optimizer, scheduler, scaler


_separator("CNN from Scratch  |  VGG_SmallSigmoid on STL-10")
t = time.time()
run_cnn_from_scratch()
print(f"\nCNN from Scratch finished in {timedelta(seconds=int(time.time() - t))}")
_clear_gpu()

---
## Experiment B — Transfer Learning
**Models:** ResNet-50 and VGG-16 (pretrained on ImageNet)  
**Modes:** Frozen backbone (feature extractor) and full fine-tune  
**Optimizer:** Adam | LR: `1e-3` frozen, `1e-4` full fine-tune  
**Scheduler:** StepLR (step=5, gamma=0.1)

In [ ]:
class STL10DatasetB(Dataset):
    def __init__(self, images: np.ndarray, labels: np.ndarray, transform=None) -> None:
        self.images    = images
        self.labels    = labels.astype(np.int64) - 1
        self.transform = transform

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        img = torch.from_numpy(self.images[idx]).permute(2, 0, 1).float() / 255.0
        if self.transform:
            img = self.transform(img)
        return img, int(self.labels[idx])


def _build_transforms_b(train: bool) -> transforms.Compose:
    norm = transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    base = [transforms.Resize(224)]
    if train:
        base += [transforms.RandomHorizontalFlip(),
                 transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2)]
    base.append(norm)
    return transforms.Compose(base)


def _load_labeled_b(images_path: str, labels_path: str, train: bool, batch_size: int) -> DataLoader:
    images  = read_all_images(images_path)
    labels  = read_labels(labels_path)
    dataset = STL10DatasetB(images, labels, transform=_build_transforms_b(train))
    return DataLoader(dataset, batch_size=batch_size, shuffle=train, num_workers=2, pin_memory=True)


def _build_resnet50(num_classes: int, feature_extract: bool) -> nn.Module:
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    if feature_extract:
        for param in model.parameters():
            param.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def _build_vgg16(num_classes: int, feature_extract: bool) -> nn.Module:
    model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
    if feature_extract:
        for param in model.features.parameters():
            param.requires_grad = False
    model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    return model


def _run_transfer_experiment(
    model: nn.Module,
    model_name: str,
    frozen: bool,
    train_loader: DataLoader,
    test_loader: DataLoader,
    num_epochs: int,
    lr: float,
) -> float:
    model      = model.to(DEVICE)
    criterion  = nn.CrossEntropyLoss()
    optimizer  = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    scheduler  = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)
    mode_label = "frozen backbone" if frozen else "full fine-tune"

    print(f"\n{'='*60}")
    print(f"  Model: {model_name}  [{mode_label}]  LR: {lr}  |  Device: {DEVICE}")
    print(f"{'='*60}")

    best_acc, final_train_acc, epochs_trained = 0.0, 0.0, 0
    for epoch in range(1, num_epochs + 1):
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            out  = model(images)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * images.size(0)
            correct    += (out.argmax(1) == labels).sum().item()
            total      += images.size(0)
        train_loss, train_acc = total_loss / total, correct / total

        model.eval()
        total_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                out  = model(images)
                loss = criterion(out, labels)
                total_loss += loss.item() * images.size(0)
                correct    += (out.argmax(1) == labels).sum().item()
                total      += images.size(0)
        test_loss, test_acc = total_loss / total, correct / total

        scheduler.step()
        epochs_trained  = epoch
        final_train_acc = train_acc
        if test_acc > best_acc:
            best_acc  = test_acc
            save_name = f"{model_name.lower().replace('-', '_')}_{'frozen' if frozen else 'full'}_best.pth"
            torch.save(model.state_dict(), save_name)

        print(
            f"Epoch [{epoch:>2}/{num_epochs}]  "
            f"Train loss: {train_loss:.4f}  acc: {train_acc*100:.2f}%  |  "
            f"Test  loss: {test_loss:.4f}  acc: {test_acc*100:.2f}%"
        )

    print(f"\nBest test accuracy — {model_name} [{mode_label}]: {best_acc*100:.2f}%")
    log_result(
        stage="B", architecture=model_name, mode=mode_label,
        optimizer="Adam", lr=lr, epochs_trained=epochs_trained,
        train_acc=final_train_acc, test_acc=best_acc,
        notes="Pretrained ImageNet weights, StepLR(step=5, gamma=0.1)",
    )
    return best_acc


print("Transfer Learning — classes defined.")

In [ ]:
def run_transfer_learning() -> None:
    BATCH_SIZE = 64
    NUM_EPOCHS = 10

    train_loader = _load_labeled_b(TRAIN_X_PATH, TRAIN_Y_PATH, train=True,  batch_size=BATCH_SIZE)
    test_loader  = _load_labeled_b(TEST_X_PATH,  TEST_Y_PATH,  train=False, batch_size=BATCH_SIZE)
    print(f"Train: {len(train_loader.dataset)} images  |  Test: {len(test_loader.dataset)} images")

    experiments = [
        ("ResNet-50", _build_resnet50, True,  1e-3),
        ("ResNet-50", _build_resnet50, False, 1e-4),
        ("VGG-16",    _build_vgg16,    True,  1e-3),
        ("VGG-16",    _build_vgg16,    False, 1e-4),
    ]

    results: list[tuple[str, str, float]] = []
    for model_name, builder, frozen, lr in experiments:
        model = builder(NUM_CLASSES, feature_extract=frozen)
        acc   = _run_transfer_experiment(model, model_name, frozen, train_loader, test_loader, NUM_EPOCHS, lr)
        results.append((model_name, "frozen" if frozen else "full fine-tune", acc))
        del model
        _clear_gpu()

    print(f"\n{'='*60}\n  Transfer Learning — Summary\n{'='*60}")
    for name, mode, acc in results:
        print(f"  {name:<12} [{mode:<15}]  best test acc: {acc*100:.2f}%")


_separator("Transfer Learning  |  ResNet-50 & VGG-16  (frozen + full fine-tune)")
t = time.time()
run_transfer_learning()
print(f"\nTransfer Learning finished in {timedelta(seconds=int(time.time() - t))}")
_clear_gpu()

---
## Experiment C — Semi-Supervised: Autoencoder + Classifier
**Step 1:** Pretrain a ConvAutoencoder on 100,000 unlabeled images (MSE loss, raw pixels)  
**Step 2:** Attach a classification head to the encoder and fine-tune on 5,000 labeled images  
**Modes:** Frozen encoder and full fine-tune

In [ ]:
class STL10LabeledDatasetC(Dataset):
    def __init__(self, images: np.ndarray, labels: np.ndarray, transform=None) -> None:
        self.images    = images
        self.labels    = labels.astype(np.int64) - 1
        self.transform = transform

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        img = torch.from_numpy(self.images[idx]).permute(2, 0, 1).float() / 255.0
        if self.transform:
            img = self.transform(img)
        return img, int(self.labels[idx])


class STL10UnlabeledDatasetC(Dataset):
    """Raw [0,1] pixels — no normalization — so MSE target matches Sigmoid decoder output."""

    def __init__(self, images: np.ndarray, transform=None) -> None:
        self.images    = images
        self.transform = transform

    def __len__(self) -> int:
        return len(self.images)

    def __getitem__(self, idx: int) -> torch.Tensor:
        img = torch.from_numpy(self.images[idx]).permute(2, 0, 1).float() / 255.0
        if self.transform:
            img = self.transform(img)
        return img


def _build_unlabeled_transforms_c() -> transforms.Compose:
    return transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
    ])


def _build_labeled_transforms_c(train: bool) -> transforms.Compose:
    norm = transforms.Normalize(mean=STL10_MEAN, std=STL10_STD)
    if train:
        return transforms.Compose([
            transforms.RandomCrop(96, padding=8),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
            norm,
        ])
    return transforms.Compose([norm])


def _load_unlabeled_c(batch_size: int) -> DataLoader:
    images  = read_all_images(UNLABELED_X_PATH)
    dataset = STL10UnlabeledDatasetC(images, transform=_build_unlabeled_transforms_c())
    return DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)


def _load_labeled_c(images_path: str, labels_path: str, train: bool, batch_size: int) -> DataLoader:
    images  = read_all_images(images_path)
    labels  = read_labels(labels_path)
    dataset = STL10LabeledDatasetC(images, labels, transform=_build_labeled_transforms_c(train))
    return DataLoader(dataset, batch_size=batch_size, shuffle=train, num_workers=2, pin_memory=True)


class ConvEncoder(nn.Module):
    """96x96x3 -> 256x6x6 via four stride-2 conv blocks."""

    def __init__(self) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,   32,  3, stride=2, padding=1), nn.BatchNorm2d(32),  nn.ReLU(),
            nn.Conv2d(32,  64,  3, stride=2, padding=1), nn.BatchNorm2d(64),  nn.ReLU(),
            nn.Conv2d(64,  128, 3, stride=2, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 256, 3, stride=2, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class ConvDecoder(nn.Module):
    """256x6x6 -> 3x96x96. Sigmoid output matches the raw [0,1] pixel input."""

    def __init__(self) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.ConvTranspose2d(128, 64,  4, stride=2, padding=1), nn.BatchNorm2d(64),  nn.ReLU(),
            nn.ConvTranspose2d(64,  32,  4, stride=2, padding=1), nn.BatchNorm2d(32),  nn.ReLU(),
            nn.ConvTranspose2d(32,  3,   4, stride=2, padding=1), nn.Sigmoid(),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)


class ConvAutoencoder(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.encoder = ConvEncoder()
        self.decoder = ConvDecoder()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.decoder(self.encoder(x))


class EncoderClassifier(nn.Module):
    """Pretrained encoder + classification head: GAP -> Linear(256->128) -> Linear(128->10)."""

    def __init__(self, encoder: ConvEncoder, num_classes: int, freeze_encoder: bool) -> None:
        super().__init__()
        self.encoder = encoder
        if freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.encoder(x))


print("Autoencoder + Classifier — classes defined.")

In [ ]:
def _pretrain_autoencoder(autoencoder: ConvAutoencoder, unlabeled_loader: DataLoader) -> float:
    AE_EPOCHS    = 30
    AE_LR        = 1e-3
    WEIGHT_DECAY = 1e-4
    PATIENCE     = 7

    autoencoder = autoencoder.to(DEVICE)
    criterion   = nn.MSELoss()
    optimizer   = optim.Adam(autoencoder.parameters(), lr=AE_LR, weight_decay=WEIGHT_DECAY)
    scheduler   = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=AE_EPOCHS)
    scaler      = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    stopper     = EarlyStopping(patience=PATIENCE)

    print(f"\n{'='*60}")
    print(f"  Autoencoder Pretraining on {len(unlabeled_loader.dataset)} unlabeled images")
    print(f"  Device: {DEVICE}")
    print(f"{'='*60}")

    final_loss, epochs_trained = 0.0, 0
    for epoch in range(1, AE_EPOCHS + 1):
        autoencoder.train()
        total_loss, total = 0.0, 0
        for images in unlabeled_loader:
            images = images.to(DEVICE)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                loss = criterion(autoencoder(images), images)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item() * images.size(0)
            total      += images.size(0)
        train_loss     = total_loss / total
        scheduler.step()
        final_loss     = train_loss
        epochs_trained = epoch
        print(f"Epoch [{epoch:>2}/{AE_EPOCHS}]  Recon loss: {train_loss:.6f}")
        if stopper(train_loss):
            print(f"\nEarly stopping at epoch {epoch}")
            break

    torch.save(autoencoder.state_dict(), "autoencoder_pretrained.pth")
    print("Autoencoder saved to autoencoder_pretrained.pth")
    log_result(
        stage="C", architecture="ConvAutoencoder", mode="pretrain (unsupervised)",
        optimizer="Adam", lr=AE_LR, epochs_trained=epochs_trained,
        train_acc=None, test_acc=None,
        notes=f"MSE on 100k unlabeled raw [0,1] pixels. Final recon loss: {final_loss:.6f}",
    )
    return final_loss


def _finetune_classifier(
    encoder: ConvEncoder,
    freeze_encoder: bool,
    train_loader: DataLoader,
    test_loader: DataLoader,
) -> float:
    CLS_EPOCHS   = 20
    WEIGHT_DECAY = 1e-4
    PATIENCE     = 7
    lr   = 1e-3 if freeze_encoder else 1e-4
    mode = "frozen encoder" if freeze_encoder else "full fine-tune"

    model     = EncoderClassifier(encoder, NUM_CLASSES, freeze_encoder).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=WEIGHT_DECAY
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CLS_EPOCHS)
    scaler    = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    stopper   = EarlyStopping(patience=PATIENCE)

    print(f"\n{'='*60}")
    print(f"  Classifier Fine-tuning  [{mode}]  LR: {lr}")
    print(f"{'='*60}")

    best_acc, final_train_acc, epochs_trained = 0.0, 0.0, 0
    for epoch in range(1, CLS_EPOCHS + 1):
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                out  = model(images)
                loss = criterion(out, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item() * images.size(0)
            correct    += (out.argmax(1) == labels).sum().item()
            total      += images.size(0)
        train_loss, train_acc = total_loss / total, correct / total

        model.eval()
        total_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                with torch.cuda.amp.autocast(enabled=USE_AMP):
                    out  = model(images)
                    loss = criterion(out, labels)
                total_loss += loss.item() * images.size(0)
                correct    += (out.argmax(1) == labels).sum().item()
                total      += images.size(0)
        test_loss, test_acc = total_loss / total, correct / total

        scheduler.step()
        epochs_trained  = epoch
        final_train_acc = train_acc
        if test_acc > best_acc:
            best_acc  = test_acc
            torch.save(model.state_dict(), f"autoenc_cls_{'frozen' if freeze_encoder else 'full'}_best.pth")

        print(
            f"Epoch [{epoch:>2}/{CLS_EPOCHS}]  "
            f"Train loss: {train_loss:.4f}  acc: {train_acc*100:.2f}%  |  "
            f"Test  loss: {test_loss:.4f}  acc: {test_acc*100:.2f}%"
        )
        if stopper(test_loss):
            print(f"\nEarly stopping at epoch {epoch}")
            break

    print(f"\nBest test accuracy [{mode}]: {best_acc*100:.2f}%")
    log_result(
        stage="C", architecture="ConvEncoder + ClassHead", mode=mode,
        optimizer="Adam", lr=lr, epochs_trained=epochs_trained,
        train_acc=final_train_acc, test_acc=best_acc,
        notes="Encoder pretrained on 100k unlabeled. Head: GAP->Linear(256->128)->Linear(128->10)",
    )
    return best_acc


def run_autoencoder_with_classifier() -> None:
    BATCH_SIZE = 64

    unlabeled_loader = _load_unlabeled_c(batch_size=BATCH_SIZE)
    train_loader     = _load_labeled_c(TRAIN_X_PATH, TRAIN_Y_PATH, train=True,  batch_size=BATCH_SIZE)
    test_loader      = _load_labeled_c(TEST_X_PATH,  TEST_Y_PATH,  train=False, batch_size=BATCH_SIZE)
    print(
        f"Unlabeled: {len(unlabeled_loader.dataset)} images  |  "
        f"Train: {len(train_loader.dataset)} labeled  |  "
        f"Test: {len(test_loader.dataset)} labeled"
    )

    autoencoder = ConvAutoencoder()
    _pretrain_autoencoder(autoencoder, unlabeled_loader)

    results: list[tuple[str, float]] = []
    for freeze in (True, False):
        fresh_encoder = ConvEncoder()
        fresh_encoder.load_state_dict(autoencoder.encoder.state_dict())
        acc = _finetune_classifier(fresh_encoder, freeze, train_loader, test_loader)
        results.append(("frozen encoder" if freeze else "full fine-tune", acc))
        del fresh_encoder
        _clear_gpu()

    print(f"\n{'='*60}\n  Autoencoder + Classifier — Summary\n{'='*60}")
    for mode, acc in results:
        print(f"  [{mode:<18}]  best test acc: {acc*100:.2f}%")


_separator("Semi-Supervised  |  Autoencoder Pretraining + Classifier Fine-tuning")
t = time.time()
run_autoencoder_with_classifier()
print(f"\nAutoencoder + Classifier finished in {timedelta(seconds=int(time.time() - t))}")
_clear_gpu()

---
## Final Summary

In [ ]:
from google.colab import files

print(f"All experiments complete. Results saved to: {RESULTS_FILE}")
files.download(RESULTS_FILE)